# step 2 진단 — granite 이상 원인 (H1 vs H2)

granite는 `S_깨끗≈0`(음수)에 회복 약함(0.12), Value 우세 안 나옴. 원인을 가른다.

- **H1 (진짜 약한 선호):** granite가 camelCase 지침을 약하게 따름 → clean에서도 camel 선호가 약함. `S_깨끗≈0`이 증거(치환 무관).
- **H2 (방법 문제):** B의 **마지막 토큰이 형태 차이 토큰이 아님**(예 `[format][M][atrix]` vs `[format][_m][atrix]` → 마지막 `atrix` 동일) → 치환이 형태를 못 바꿈.

이 노트북: (1) 토크나이저로 이름 토큰화·**어느 토큰이 다른지** 확인(H2), (2) clean 조건에서 granite가 **실제로 camel을 생성하는지** 확인(H1).


In [ ]:
!pip install -q transformers accelerate torch
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step2/granite-3b-code
!git checkout step2/granite-3b-code
!git pull --quiet origin step2/granite-3b-code
!pip install -e . -q
import sys; sys.path.insert(0,'src')

In [ ]:
# H2 진단 — granite 토크나이저: 이름/후보가 어떻게 쪼개지고 어느 토큰이 다른가
from transformers import AutoTokenizer
from harness.tasks import NAME_PAIR_POOL
from harness.conditions import Notation

tok = AutoTokenizer.from_pretrained('ibm-granite/granite-3b-code-instruct-2k')

def toks(s):
    ids = tok(s, add_special_tokens=False)['input_ids']
    return [tok.decode([i]) for i in ids]

print('=== 이름 6쌍: def <name>( 문맥에서 토큰화 (하네스와 동일) ===')
last_differs = same_last = 0
for t in NAME_PAIR_POOL[:8]:
    sn, cm = t.name(Notation.SNAKE), t.name(Notation.CAMEL)
    st, ct = toks(f'def {sn}('), toks(f'def {cm}(')
    # 'def ' 이후 ~ '(' 이전이 이름부. 마지막 이름 토큰은 '(' 바로 앞.
    s_last, c_last = st[-2], ct[-2]   # [-1]은 '('
    same = (s_last == c_last)
    same_last += same; last_differs += (not same)
    print(f'  {sn:18} -> {st}')
    print(f'  {cm:18} -> {ct}')
    print(f'     마지막 이름토큰:  snake={s_last!r}  camel={c_last!r}  ->', '동일(!! B 무효)' if same else '다름(B 유효)')
    print()
print(f'요약: 마지막 토큰이 다른 쌍 {last_differs} / 같은 쌍 {same_last}  (같은 쪽이 많으면 H2)')

print('\n=== 채점 후보 토큰화 (S_깨끗 계산에 쓰임) ===')
for s in ['removeDuplicates','remove_duplicates']:
    print(f'  {s:18} -> {toks(s)}')

In [ ]:
# H1 진단 — clean 조건(전부 camel 선행 + camel 지침)에서 granite가 실제로 camel을 쓰나
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)
from harness.model import load_model
from harness import run

MODEL = ModelSpec(name='ibm-granite/granite-3b-code-instruct-2k', family='granite', dtype='float16')
handle = load_model(MODEL)

def clean_cond(s):   # n_compliant=12 = 선행 전부 준수(camel), 지침 camel = 가장 쉬운 조건
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=12, n_functions=12, composition=Composition.POOL),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL), seed=s)

print('clean(전부 camel 선행+camel 지침)에서 granite가 생성한 첫 함수 표기:')
n_camel=0
for s in range(5):
    out = run(clean_cond(s), handle=handle, max_new_tokens=64)
    e = out.metrics.extra
    print(f'  seed{s}: 표기={e["turn_notations"][0]:6}  이름={e["turn_names"][0]}')
    n_camel += (e['turn_notations'][0]=='camel')
print(f'\n=> camel 생성 {n_camel}/5.  4~5면 H1 기각(지침 따름, 문제는 방법/측정), 0~1이면 H1(약한 선호) 지지.')